In [1]:
import json
from pymongo import MongoClient
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from joblib import Parallel, delayed
import nltk

# Télécharger les corpus nécessaires pour NLTK
nltk.download('punkt')

# Vérifier si pyarabic est installé
try:
    import pyarabic
except ImportError:
    print("La bibliothèque 'pyarabic' est requise pour le tokenizer arabe.")
    print("Veuillez l’installer avec : pip install pyarabic")
    exit(1)

# Fonction pour récupérer les données depuis MongoDB
def get_data_from_mongodb():
    client = MongoClient('mongodb://localhost:27017')
    db = client['dataset']  # Nom de ta base
    collection = db['LesArticles']  # Nom de la collection
    documents = list(collection.find({
        'cluster_id': {'$exists': True},
        'content': {'$exists': True, '$ne': ''}
    }))
    return db, documents

# Fonction pour générer un résumé avec LexRank
def generate_summary(text, sentences_count=5):  # ← augmenté à 5 phrases
    try:
        if not text or not isinstance(text, str):
            return "النص غير صالح أو فارغ"
        
        # ↑ Limiter à 4000 caractères pour un résumé plus riche
        text = text[:4000]
        
        parser = PlaintextParser.from_string(text, Tokenizer("arabic"))
        summarizer = LexRankSummarizer()
        summary = summarizer(parser.document, sentences_count)
        return " ".join(str(sentence) for sentence in summary)
    except Exception as e:
        print(f"Erreur lors du résumé : {e}")
        return "حدث خطأ أثناء إنشاء الملخص"

# Fonction principale de clustering et génération des résumés
def cluster_and_generate_summary(documents):
    clusters = {}
    for doc in documents:
        cluster_id = doc['cluster_id']
        content = doc['content']
        if cluster_id not in clusters:
            clusters[cluster_id] = ""
        clusters[cluster_id] += content + " "

    results = Parallel(n_jobs=-1)(
        delayed(lambda cid, txt: (cid, generate_summary(txt)))(cid, text)
        for cid, text in clusters.items()
    )
    cluster_summaries = dict(results)

    with open('cluster_summaries_LexRank.json', 'w', encoding='utf-8') as f:
        json.dump(cluster_summaries, f, ensure_ascii=False, indent=4)

    print("✅ تم حفظ ملخصات العناقيد في ملف 'cluster_summaries_LexRank.json'.")

# Lancement
if __name__ == "__main__":
    print("🚀 بدء تنفيذ البرنامج...")
    db, documents = get_data_from_mongodb()
    if documents:
        cluster_and_generate_summary(documents)
    else:
        print("⚠️ لم يتم العثور على أي وثائق في قاعدة البيانات.")
    print("✅ تم الانتهاء من البرنامج بنجاح.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hajar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🚀 بدء تنفيذ البرنامج...
✅ تم حفظ ملخصات العناقيد في ملف 'cluster_summaries_LexRank.json'.
✅ تم الانتهاء من البرنامج بنجاح.
